# MyFitnessPal weight journey and TDEE scenarios

Reproducible dashboard from the CSV files in `myfitnesspal/data`. Withings activity is optional. Missing diary entries are **unknown intake**, not zero.

In [ ]:
from pathlib import Path
import os, json, time, warnings
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fitness_analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open notebook inside sandbox-scripts repository")
load_dotenv(ROOT / ".env", override=False)
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from fitness_analysis import load_mfp, load_withings, join_data, monthly_summary, forecast, lean_goal_table
START = "2026-04-01"
END = "2026-10-20"
OUT = ROOT / "myfitnesspal" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (11, 4)
print("Repository:", ROOT, "\\nWindow:", START, "to", END)


## 1. Data integrity and input coverage

In [ ]:
daily,weights=load_mfp(ROOT/"myfitnesspal"/"data")
activity=load_withings(ROOT/"withings"/"data")
frame=join_data(daily,weights,activity)
print("Food days:",len(daily),"logged:",int(daily.logged.sum()),
      "possibly partial:",int(daily.possibly_partial.sum()))
print("Weight rows:",len(weights),"changes in exported weight:",int(weights.changed_from_previous.sum()))
display(monthly_summary(frame))
display(frame.tail(10))

## 2. Recorded weight history

Unchanged repeated chart values are not assumed to be new weigh-ins.

In [ ]:
ax=frame.plot(x="date",y=["weight_kg","weight_7d_mean"],title="Recorded weight and rolling mean")
ax.set_ylabel("kg");plt.tight_layout();plt.show()

## 3. Food logging and macro nutrients

Sub-1,400 kcal logs are marked potentially incomplete; adjust threshold for your actual meal pattern.

In [ ]:
fig,ax=plt.subplots()
for status,sub in daily.groupby("plausibly_complete"):
    ax.scatter(sub.date,sub.calories,label="Likely complete" if status else "Untracked/possibly partial",s=15)
ax.legend();ax.set_title("Calories by log completeness");ax.set_ylabel("kcal");plt.tight_layout();plt.show()
daily[["date","calories","protein_g","carbs_g","fat_g"]].set_index("date").rolling(7,min_periods=3).mean().plot(title="7-day recorded dietary averages")
plt.tight_layout();plt.show()

## 4. Withings energy and movement (when exported)

In [ ]:
if not activity.empty:
    display(activity[["date","tdee_calories","active_burned_calories","steps"]].describe(include="all"))
    activity.plot(x="date",y=[x for x in ["tdee_calories","active_burned_calories"] if x in activity],title="Withings energy expenditure")
    plt.tight_layout();plt.show()
    activity.plot(x="date",y="steps",title="Steps per day");plt.tight_layout();plt.show()
else:
    print("No Withings CSV found; forecasts remain available without it.")

## 5. Intake versus expenditure

Only show calculated recorded-day deficits on plausibly complete food days; this is **not** actual deficit on missing diary days.

In [ ]:
if "tdee_calories" in frame:
    observed=frame.dropna(subset=["observed_deficit_kcal"])
    print("Overlapping plausible days:",len(observed))
    display(observed[["date","calories","tdee_calories","observed_deficit_kcal"]].head(15))
    if len(observed):
        observed.plot(x="date",y="observed_deficit_kcal",title="Reported-day energy deficits (partial sample)")
        plt.axhline(0,color="black",linewidth=.8);plt.tight_layout();plt.show()

## 6. Cut milestones and scenario sensitivity

In [ ]:
scenarios=[]
for assumed_tdee in [2450,2600,2750]:
    trajectory,milestones=forecast(start_weight=86.8,start_date=END,tdee=assumed_tdee)
    scenarios.append(trajectory.assign(scenario=f"TDEE {assumed_tdee}"))
    print(f"Initial TDEE {assumed_tdee} kcal/day")
    display(milestones)
fig,ax=plt.subplots()
for s in scenarios:
    ax.plot(s["date"],s["weight_kg"],label=s["scenario"].iloc[0])
for goal in (84,82,80,78,76):
    ax.axhline(goal,linewidth=.5,linestyle=":",color="grey")
ax.set_title("Weight loss scenarios: 1,800 baseline + two 600-kcal meal extras weekly")
ax.set_ylabel("kg");ax.legend();plt.tight_layout();plt.show()
display(lean_goal_table())
print("Maintenance through April 2027 (illustrative):",
      {kg:round(2600+22*(kg-86.8)) for kg in [84,82,80,78,76]})


## 7. Maintenance until April 2027

In [ ]:
maintenance=pd.DataFrame({"weight_kg":[84,82,80,78,76]})
maintenance["estimated_maintenance_kcal"]=2600+22*(maintenance.weight_kg-86.8)
maintenance["lean_gain_start_kcal"]=maintenance.estimated_maintenance_kcal+200
display(maintenance.round())
maintenance.plot(x="weight_kg",y=["estimated_maintenance_kcal","lean_gain_start_kcal"],title="Planning targets by weight")
plt.ylabel("kcal/day");plt.tight_layout();plt.show()

## 8. Long-term 95 kg composition goals

September 2026 DEXA-derived fat-free mass 78.021 kg; growth-rate assumptions are exploratory, not predictive of individual muscle potential.

In [ ]:
goals=lean_goal_table(78.021)
display(goals.round(2))
rates=[.5,1,1.5,2,3]
sens=pd.DataFrame({"annual_ffm_gain_kg":rates})
for _,goal in goals.iterrows():
    sens[goal["goal"]+" years"]=goal["additional_ffm_kg"]/sens["annual_ffm_gain_kg"]
display(sens.round(1))
sens.set_index("annual_ffm_gain_kg").plot(title="Years to fat-free-mass goal by assumed rate",marker="o")
plt.ylabel("years");plt.tight_layout();plt.show()

## 9. Export dashboard data

Tables and calculated scenarios are saved locally; plots render directly within Jupyter.

In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
frame.to_csv(OUT/"combined_weight_diet_activity.csv",index=False)
monthly_summary(frame).to_csv(OUT/"monthly_summary.csv",index=False)
goals.to_csv(OUT/"lean_goal_scenarios.csv",index=False)
print("Saved dashboards to",OUT)